# Explore the Airescents data

Use this notebook to look at the data yourself and **check the app's answers**.

**How to use it**
- Run a cell with **Shift+Enter**. The first time, VS Code asks you to **select a kernel**: choose **Python Environments**, then the suggested Python.
- Run the setup cell first (the one right below), then any other cell in any order.
- New to SQL? Read *Reading SQL in 15 Minutes* first. Every query here has a plain-English note above it.

In [ ]:
import duckdb
db = duckdb.connect("../data/airescents.duckdb", read_only=True)

def q(sql):
    """Run a SQL query and show the result as a table."""
    return db.sql(sql).df()

## What's in the database

In [ ]:
q("SHOW TABLES")

In [ ]:
q("SELECT * FROM campaigns")

In [ ]:
q("SELECT * FROM creatives")

In [ ]:
q("SELECT * FROM ad_performance LIMIT 10")

In [ ]:
q("SELECT * FROM budgets LIMIT 12")

In [ ]:
q("SELECT * FROM store_orders LIMIT 10")

## Checking the team's questions

For each question: ask the app first, then run the query below and compare. Do they match?
If not, open **See the SQL** under the app's answer and look for the difference.

### 1. How much did we spend on ads in total in 2025?
*What this query does:* Adds up the `spend_usd` column for every row.

In [ ]:
q("""
SELECT ROUND(SUM(spend_usd), 2) AS total_spend_usd
FROM ad_performance
""")

### 2. Which platform got the most clicks in Q2?
*What this query does:* Keeps only April 1 to June 30, adds up clicks for each platform, and shows the biggest first.

In [ ]:
q("""
SELECT platform, SUM(clicks) AS clicks
FROM ad_performance
WHERE date BETWEEN '2025-04-01' AND '2025-06-30'
GROUP BY platform
ORDER BY clicks DESC
""")

### 3. How many campaigns ran on TikTok?
*What this query does:* Counts the rows in the campaigns table where the platform is TikTok.

In [ ]:
q("""
SELECT COUNT(*) AS tiktok_campaigns
FROM campaigns
WHERE platform = 'TikTok'
""")

### 4. What was Meta's click-through rate in July?
*What this query does:* Keeps Meta in July, then divides total clicks by total impressions and turns it into a percent.

In [ ]:
q("""
SELECT ROUND(100.0 * SUM(clicks) / SUM(impressions), 2) AS ctr_pct
FROM ad_performance
WHERE platform = 'Meta' AND date BETWEEN '2025-07-01' AND '2025-07-31'
""")

### 5. What was the CPA by platform in Q3?
*What this query does:* Keeps July 1 to September 30, then for each platform divides total spend by total conversions.

In [ ]:
q("""
SELECT platform, ROUND(SUM(spend_usd) / SUM(conversions), 2) AS cpa_usd
FROM ad_performance
WHERE date BETWEEN '2025-07-01' AND '2025-09-30'
GROUP BY platform
ORDER BY cpa_usd
""")

### 6. What was our overall ROAS in November?
*What this query does:* Keeps November, then divides total revenue by total spend.

In [ ]:
q("""
SELECT ROUND(SUM(revenue_usd) / SUM(spend_usd), 2) AS roas
FROM ad_performance
WHERE date BETWEEN '2025-11-01' AND '2025-11-30'
""")

### 7. What was the conversion rate for retargeting vs prospecting?
*What this query does:* The daily results don't say what each campaign's objective is, so it JOINs them to the campaigns table, then divides conversions by clicks for each objective.

In [ ]:
q("""
SELECT c.objective, ROUND(100.0 * SUM(a.conversions) / SUM(a.clicks), 2) AS conversion_rate_pct
FROM ad_performance a
JOIN campaigns c ON a.campaign_id = c.campaign_id
WHERE c.objective IN ('Retargeting', 'Prospecting')
GROUP BY c.objective
""")

### 8. How much did spend change from November to December, as a percent?
*What this query does:* Adds up November's spend and December's spend separately (FILTER keeps only the rows for that month), then works out the percent change.

In [ ]:
q("""
SELECT
  ROUND(SUM(spend_usd) FILTER (WHERE month(date) = 11), 2) AS november_spend,
  ROUND(SUM(spend_usd) FILTER (WHERE month(date) = 12), 2) AS december_spend,
  ROUND(100.0 * (SUM(spend_usd) FILTER (WHERE month(date) = 12)
               / SUM(spend_usd) FILTER (WHERE month(date) = 11) - 1), 1) AS pct_change
FROM ad_performance
""")

### 9. Which 3 campaigns had the lowest CPA among those spending at least $5,000?
*What this query does:* Works out spend and CPA for each campaign. HAVING keeps only campaigns that spent $5,000 or more (it's like WHERE, but for totals). Then it sorts by CPA, lowest first, and keeps 3.

In [ ]:
q("""
SELECT c.campaign_name,
       ROUND(SUM(a.spend_usd), 2) AS spend_usd,
       ROUND(SUM(a.spend_usd) / SUM(a.conversions), 2) AS cpa_usd
FROM ad_performance a
JOIN campaigns c ON a.campaign_id = c.campaign_id
GROUP BY c.campaign_name
HAVING SUM(a.spend_usd) >= 5000
ORDER BY cpa_usd
LIMIT 3
""")

### 10. Which week of the year had the highest spend?
*What this query does:* date_trunc('week', date) turns every date into the Monday that starts its week, so spend can be added up per week. Then it keeps the biggest one.

In [ ]:
q("""
SELECT date_trunc('week', date) AS week_starting, ROUND(SUM(spend_usd), 2) AS spend_usd
FROM ad_performance
GROUP BY week_starting
ORDER BY spend_usd DESC
LIMIT 1
""")

### 11. How did Google Search's actual spend compare to its budget each month?
*What this query does:* Matches each month's budget to that month's actual Google Search spend, then shows actual spend as a percent of budget.

In [ ]:
q("""
SELECT b.month,
       b.planned_budget_usd,
       ROUND(SUM(a.spend_usd), 2) AS actual_spend_usd,
       ROUND(100.0 * SUM(a.spend_usd) / b.planned_budget_usd, 1) AS pct_of_budget
FROM budgets b
JOIN ad_performance a
  ON a.platform = b.platform AND date_trunc('month', a.date) = b.month
WHERE b.platform = 'Google Search'
GROUP BY b.month, b.planned_budget_usd
ORDER BY b.month
""")

### 12. Which creative format has the best CTR on Meta?
*What this query does:* JOINs the daily results to the creatives table to get each ad's format (Video, Static, Carousel), then works out click-through rate for each format on Meta.

In [ ]:
q("""
SELECT cr.format, ROUND(100.0 * SUM(a.clicks) / SUM(a.impressions), 2) AS ctr_pct
FROM ad_performance a
JOIN creatives cr ON a.ad_id = cr.ad_id
WHERE a.platform = 'Meta'
GROUP BY cr.format
ORDER BY ctr_pct DESC
""")

### 13. How many conversions did we get in 2025?
*What this query does:* Conversions are recorded in two places: the ad platforms report their own, and the store records actual orders. This query counts both: what the platforms reported, store orders whose last click was an ad, and all store orders.

In [ ]:
q("""
SELECT
  (SELECT SUM(conversions) FROM ad_performance) AS conversions_reported_by_ad_platforms,
  (SELECT SUM(orders) FROM store_orders
   WHERE last_click_channel IN ('Meta', 'Google Search', 'TikTok', 'YouTube')) AS store_orders_from_ads,
  (SELECT SUM(orders) FROM store_orders) AS store_orders_total
""")

## Questions 14–15: try these yourself

No pre-written queries here. Write your own, or ask Claude Code: *"Write a SQL query for my notebook that answers: ..."*
Then compare your result with the app's answer. If they disagree, try to work out why.

### 14. What was Meta's CPA in the week of August 12?

In [ ]:
q("""

""")

### 15. What was our ROAS on Snapchat?

In [ ]:
q("""

""")

### 16. Which 5 ads had the best click-through rate?
*What this query does:* JOINs the daily results to the creatives table to get each ad's name, works out click-through rate (total clicks ÷ total impressions) for each ad, and keeps the top 5.

In [ ]:
q("""
SELECT cr.ad_name, a.platform,
       ROUND(100.0 * SUM(a.clicks) / SUM(a.impressions), 2) AS ctr_pct
FROM ad_performance a
JOIN creatives cr ON a.ad_id = cr.ad_id
GROUP BY cr.ad_name, a.platform
ORDER BY ctr_pct DESC
LIMIT 5
""")

## Your own queries

Anything you're curious about. Write a query between the triple quotes and run it.

In [ ]:
q("""

""")